# 🚑 Train the Smart Intersection AI model (free Colab GPU)

This notebook trains YOLOv8 to recognise **ambulance**, **police**, **car** and
**motorcycle** so `main.py` can do a *real* emergency override instead of the
simulated one.

**Before you start:** set `Runtime → Change runtime type → T4 GPU`.

Workflow (same as the repo, just on Colab):
1. Get images (Roboflow "Ambulance detection" datasets are the fastest route, or
   upload your own frames from `prepare_dataset.py frames`).
2. Label: every ambulance, police car, car, motorcycle gets a box.
3. Train here → download `best.pt` → drop it in `models/best.pt` in the repo.


In [ ]:
!nvidia-smi
!pip install -q ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| cuda:", torch.cuda.is_available())

## 1. Get the dataset

**Option A (fastest): Roboflow.** Create a free account, find an ambulance /
emergency-vehicle dataset with a YOLOv8 export, then paste the download snippet:

```python
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_KEY")
project = rf.workspace("...").project("...")
dataset = project.version(1).download("yolov8")
```

**Option B: your own data.** Upload a zip of `dataset/` (built locally with
`prepare_dataset.py`) and unzip it.

In [ ]:
# Option A
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# dataset = rf.workspace("YOUR_WS").project("YOUR_PROJECT").version(1).download("yolov8")

# Option B — upload your own zip
# from google.colab import files
# up = files.upload()            # pick dataset.zip
# !unzip -q -o dataset.zip -d data

import os
print(os.listdir("."))


## 2. Write `data.yaml` for **our** 4 classes

Two things matter:

* `nc: 4` and the exact class order **0 ambulance, 1 police, 2 car, 3 motorcycle**
* the `path:` must point at the folder that actually exists on Colab.

If you use a public Roboflow dataset, its class names/order are probably
different — either remap them below or keep Roboflow's `data.yaml` and change
`PRIORITY_KEYWORDS` matching (our detector matches on *names*, not ids, so
`'ambulance'` still works whatever the index is).

In [ ]:
DATA_YAML = """path: /content/data
train: images/train
val: images/val
nc: 4
names:
  0: ambulance
  1: police
  2: car
  3: motorcycle
"""
open("data.yaml", "w").write(DATA_YAML)
print(DATA_YAML)

# sanity check: does the folder match?
for split in ("train", "val"):
    p = f"/content/data/images/{split}"
    print(split, "->", len(os.listdir(p)) if os.path.isdir(p) else "MISSING")

## 3. Train

100 epochs on a T4 with ~1–3k images ≈ 40–90 minutes. Start with 50 epochs if
you are short on time; the model is small (yolov8n, ~3M params).

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=25,
    name="smart_traffic_model",
    hsv_v=0.5, hsv_s=0.7,     # night-time footage
    plots=True,
)

## 4. Check the numbers before you trust it

In [ ]:
metrics = model.val()
print("mAP50   ", round(float(metrics.box.map50), 4))
print("mAP50-95", round(float(metrics.box.map), 4))
for i, name in metrics.names.items():
    m = metrics.box.ap50
    print(f"  class {i} {name:<12} AP50 {float(m[i]):.3f}" if i < len(m) else "")

**Read this like an engineer:** overall mAP can look great while `ambulance`
is still bad, because cars dominate the dataset. Always check the per-class
line for `ambulance` — that is the class the whole project depends on.
If `ambulance` AP50 < 0.6, collect more ambulance images (especially Indian
108 / EMRI ambulances, night shots, and rear/side views).

## 5. Look at a prediction

In [ ]:
from ultralytics import YOLO
from glob import glob
from IPython.display import Image, display

best = YOLO("runs/detect/smart_traffic_model/weights/best.pt")
imgs = glob("/content/data/images/val/*.jpg")[:3]
best.predict(imgs, save=True, conf=0.35, project="preds", name="check")
for p in glob("preds/check/*.jpg"):
    display(Image(p))

## 6. Download `best.pt` and use it in the repo

In [ ]:
from google.colab import files
import shutil, os
os.makedirs("models", exist_ok=True)
shutil.copy("runs/detect/smart_traffic_model/weights/best.pt", "models/best.pt")
files.download("models/best.pt")
print("put best.pt into Priority_Traffic_router/models/ then run: python main.py")